# Cross-topic Argument Mining with RoBERTa — Colab (end-to-end)

Re-implementation of Stab et al. (EMNLP 2018), *Cross-topic Argument Mining from Heterogeneous Sources*, with **RoBERTa replacing the Contextual BiLSTM (BiCLSTM)**.

Run the cells top-to-bottom. The only cell you need to edit is **Section C (Configuration)** — paths, hyperparameters, and seeds.

**Sections**
- A. Mount Drive
- B. Install dependencies
- C. **Configuration** (edit me)
- D. Imports
- E. Data — UKP loader
- F. Data — DIP2016 loader
- G. Tokenization & collation
- H. Model — single-task RoBERTa
- I. Model — RoBERTa MTL (shared encoder + UKP head + DIP head)
- J. Metrics (Table 4 of the paper)
- K. Training routines (single-task + MTL)
- L. Sanity check
- M. Smoke test (1 topic, 1 seed, 2 epochs)
- N. Full grid (resumable, writes per-run JSONs to Drive)
- O. Inspect summary


## A. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## B. Install dependencies

In [ ]:
!pip install -q "transformers>=4.41" "scikit-learn>=1.3" "pandas>=2.0" "tqdm>=4.66"

## C. Configuration  ← edit me

Every path, hyperparameter, and seed lives here.  Nothing below this cell needs editing for normal runs.

In [ ]:
from pathlib import Path

# ----- Paths on Google Drive (absolute) -----
UKP_CSV    = Path('/content/drive/MyDrive/PhD Ali 26/Dataset/UKP csv/8 UKP datasets.csv')
DIP_DIR    = Path('/content/drive/MyDrive/PhD Ali 26/Dataset/DIP2016')
OUTPUT_DIR = Path('/content/drive/MyDrive/PhD Ali 26/results/Ro2026')

# Optional: CSV with columns [queryID, query_text] for DIP MTL.
# Leave as None if you don't have one (queryID itself will be used as text).
QUERY_TEXT_CSV: Path | None = None

# ----- Model / tokenization -----
MODEL_NAME = 'roberta-base'
MAX_LENGTH = 128

# ----- Optimization -----
EPOCHS           = 10
BATCH_SIZE       = 32
EVAL_BATCH_SIZE  = 64
LR               = 2e-5
WEIGHT_DECAY     = 0.01
WARMUP_RATIO     = 0.06
GRAD_CLIP        = 1.0
NUM_WORKERS      = 2

# ----- Experimental protocol -----
SEEDS         = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]   # paper uses 10 seeds
LABEL_SETUPS  = [2, 3]                            # 2-label and 3-label
MTL_MODES     = [False, True]                     # False = single-task; True = MTL+DIP2016
TEST_TOPICS: list[str] | None = None              # None = all 8 paper topics; or e.g. ['gun control']

# ----- MTL specifics -----
DIP_MAX_EXAMPLES = 300_000   # paper: 300K of 600K. Set None to use all.

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert UKP_CSV.exists(),  f'UKP file not found: {UKP_CSV}'
assert DIP_DIR.exists(),  f'DIP folder not found: {DIP_DIR}'
print('UKP   :', UKP_CSV)
print('DIP   :', DIP_DIR)
print('OUT   :', OUTPUT_DIR)

## D. Imports

In [ ]:
import json
import random
import xml.etree.ElementTree as ET
from dataclasses import dataclass, field, replace
from functools import partial
from typing import Iterable

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, precision_recall_fscore_support
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    RobertaForSequenceClassification,
    RobertaModel,
    RobertaTokenizerFast,
    get_linear_schedule_with_warmup,
)

TOPICS = [
    'abortion', 'cloning', 'death penalty', 'gun control',
    'marijuana legalization', 'minimum wage', 'nuclear energy', 'school uniforms',
]
LABELS_3 = ['NoArgument', 'Argument_against', 'Argument_for']
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

## E. Data — UKP loader

UKP is a single CSV with columns:
`topic, retrievedUrl, archivedUrl, sentenceHash, sentence, annotation, set` where
`annotation ∈ {NoArgument, Argument_for, Argument_against}` and
`set ∈ {train, val, test}`.

In [ ]:
@dataclass
class Example:
    topic: str
    sentence: str
    label: int

def label_to_id(annotation: str, num_labels: int) -> int:
    if num_labels == 3:
        return LABELS_3.index(annotation)
    return 0 if annotation == 'NoArgument' else 1

def load_ukp_csv(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    missing = {'topic', 'sentence', 'annotation', 'set'} - set(df.columns)
    if missing:
        raise ValueError(f'UKP CSV missing columns: {missing}')
    df['topic_norm'] = df['topic'].astype(str).str.strip().str.lower()
    return df

def build_ukp_splits(csv_path: Path, test_topic: str, num_labels: int) -> dict[str, list[Example]]:
    df = load_ukp_csv(csv_path)
    t = test_topic.strip().lower()
    if t not in df['topic_norm'].unique():
        raise ValueError(f'Topic {test_topic!r} not in CSV. Available: {sorted(df["topic_norm"].unique())}')
    train, val, test = [], [], []
    for _, row in df.iterrows():
        ex = Example(
            topic=str(row['topic']),
            sentence=str(row['sentence']),
            label=label_to_id(str(row['annotation']), num_labels),
        )
        if row['topic_norm'] == t:
            if row['set'] == 'test':
                test.append(ex)
        else:
            if row['set'] == 'train':
                train.append(ex)
            elif row['set'] == 'val':
                val.append(ex)
    return {'train': train, 'val': val, 'test': test}

class UKPDataset(Dataset):
    def __init__(self, examples: Iterable[Example], tokenizer, max_length: int = 128):
        self.examples = list(examples)
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self) -> int:
        return len(self.examples)
    def __getitem__(self, idx: int) -> dict:
        ex = self.examples[idx]
        enc = self.tokenizer(ex.topic, ex.sentence,
                             truncation=True, max_length=self.max_length, padding=False)
        item = {k: torch.as_tensor(v) for k, v in enc.items()}
        item['labels'] = torch.as_tensor(ex.label, dtype=torch.long)
        return item

## F. Data — DIP2016 loader

Parses XML files of shape `<singleQueryResults queryID=…><documents><document><sentences><s relevant="true|false"><content>…`.
Uses the `queryID` itself as the topic text when no `queryID → query_text` mapping is supplied.

In [ ]:
@dataclass
class DIPExample:
    query: str
    sentence: str
    label: int  # 1 = relevant, 0 = not relevant

def _parse_dip_xml(path: Path, query_text_map: dict[str, str] | None = None) -> list[DIPExample]:
    root = ET.parse(path).getroot()
    qid = root.attrib.get('queryID', path.stem)
    qtext = (query_text_map.get(str(qid)) if query_text_map else None) or str(qid)
    out: list[DIPExample] = []
    for s in root.iter('s'):
        rel = s.attrib.get('relevant', 'false').strip().lower()
        c = s.find('content')
        if c is None or c.text is None:
            continue
        text = c.text.strip()
        if not text:
            continue
        out.append(DIPExample(query=qtext, sentence=text, label=(1 if rel == 'true' else 0)))
    return out

def load_dip2016(dip_dir: Path,
                 query_text_map: dict[str, str] | None = None,
                 limit_files: int | None = None,
                 max_examples: int | None = None) -> list[DIPExample]:
    files = sorted(dip_dir.glob('*.xml'))
    if limit_files is not None:
        files = files[:limit_files]
    out: list[DIPExample] = []
    for f in files:
        out.extend(_parse_dip_xml(f, query_text_map))
        if max_examples is not None and len(out) >= max_examples:
            return out[:max_examples]
    return out

class DIPDataset(Dataset):
    def __init__(self, examples: Iterable[DIPExample], tokenizer, max_length: int = 128):
        self.examples = list(examples)
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self) -> int:
        return len(self.examples)
    def __getitem__(self, idx: int) -> dict:
        ex = self.examples[idx]
        enc = self.tokenizer(ex.query, ex.sentence,
                             truncation=True, max_length=self.max_length, padding=False)
        item = {k: torch.as_tensor(v) for k, v in enc.items()}
        item['labels'] = torch.as_tensor(ex.label, dtype=torch.long)
        return item

# Optional queryID -> query_text map
QUERY_TEXT_MAP: dict[str, str] = {}
if QUERY_TEXT_CSV is not None and Path(QUERY_TEXT_CSV).exists():
    qdf = pd.read_csv(QUERY_TEXT_CSV)
    QUERY_TEXT_MAP = {str(q): str(t) for q, t in zip(qdf['queryID'], qdf['query_text'])}
    print(f'Loaded {len(QUERY_TEXT_MAP)} queryID->text mappings')
else:
    print('No queryID->text mapping; using queryID as text for DIP.')

## G. Collation

In [ ]:
def collate(batch: list[dict], pad_token_id: int) -> dict:
    max_len = max(x['input_ids'].size(0) for x in batch)
    out = {}
    for key in ('input_ids', 'attention_mask'):
        if key not in batch[0]:
            continue
        pad_val = pad_token_id if key == 'input_ids' else 0
        stacked = torch.full((len(batch), max_len), pad_val, dtype=torch.long)
        for i, x in enumerate(batch):
            stacked[i, :x[key].size(0)] = x[key]
        out[key] = stacked
    out['labels'] = torch.stack([x['labels'] for x in batch])
    return out

## H. Model — single-task RoBERTa

Topic information is injected by encoding `(topic, sentence)` as a RoBERTa sentence pair — the transformer analogue of the paper's i-/c-gate topic injection in the BiCLSTM.

In [ ]:
def build_single_task_model(model_name: str, num_labels: int):
    tokenizer = RobertaTokenizerFast.from_pretrained(model_name)
    model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
    return model, tokenizer

## I. Model — RoBERTa MTL  (shared encoder + UKP head + DIP head)

Replaces `mtl+biclstm+dip2016` from the paper.

In [ ]:
class RobertaMTL(nn.Module):
    def __init__(self, model_name: str, main_num_labels: int,
                 aux_num_labels: int = 2, dropout: float = 0.1):
        super().__init__()
        self.encoder = RobertaModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.main_head = nn.Linear(hidden, main_num_labels)
        self.aux_head  = nn.Linear(hidden, aux_num_labels)
        self.loss_fct  = nn.CrossEntropyLoss()
    def forward(self, input_ids, attention_mask, labels=None, task: str = 'main'):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(out.last_hidden_state[:, 0, :])  # <s> token
        head = self.main_head if task == 'main' else self.aux_head
        logits = head(pooled)
        loss = self.loss_fct(logits, labels) if labels is not None else None
        return {'loss': loss, 'logits': logits}

def build_mtl_model(model_name: str, main_num_labels: int):
    tokenizer = RobertaTokenizerFast.from_pretrained(model_name)
    model = RobertaMTL(model_name, main_num_labels=main_num_labels)
    return model, tokenizer

## J. Metrics  (Table 4 of Stab et al.)

- Macro-F1
- 2-label: `P_arg`, `R_arg`
- 3-label: `P_arg+`, `R_arg+`, `P_arg-`, `R_arg-`

In [ ]:
def compute_metrics(y_true, y_pred, num_labels: int) -> dict:
    out: dict[str, float] = {}
    out['macro_f1'] = float(f1_score(y_true, y_pred, average='macro', zero_division=0))
    if num_labels == 2:
        p, r, _, _ = precision_recall_fscore_support(y_true, y_pred, labels=[1], zero_division=0)
        out['P_arg'] = float(p[0]); out['R_arg'] = float(r[0])
    else:
        labels = [LABELS_3.index('Argument_for'), LABELS_3.index('Argument_against')]
        p, r, _, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
        out['P_arg+'] = float(p[0]); out['R_arg+'] = float(r[0])
        out['P_arg-'] = float(p[1]); out['R_arg-'] = float(r[1])
    return out

def aggregate_runs(runs: list[dict]) -> dict:
    keys = [k for k in runs[0].keys() if isinstance(runs[0][k], (int, float))]
    return {k: (float(np.mean([r[k] for r in runs])), float(np.std([r[k] for r in runs]))) for k in keys}

## K. Training routines

Per-epoch checkpoints are written to `OUTPUT_DIR/checkpoints/<tag>/ckpt.pt` and overwritten in place (single slot). If Colab disconnects mid-run, just re-execute Section N — the in-flight run resumes at the next epoch with its optimizer, scheduler, and RNG state restored; runs whose final JSON already exists are skipped entirely. Each ckpt is ~1–1.5 GB and is deleted once the run completes and the final JSON has been written.

In [ ]:
@dataclass
class HParams:
    ukp_csv: Path = UKP_CSV
    dip_dir: Path = DIP_DIR
    output_dir: Path = OUTPUT_DIR
    model_name: str = MODEL_NAME
    max_length: int = MAX_LENGTH
    epochs: int = EPOCHS
    batch_size: int = BATCH_SIZE
    eval_batch_size: int = EVAL_BATCH_SIZE
    lr: float = LR
    weight_decay: float = WEIGHT_DECAY
    warmup_ratio: float = WARMUP_RATIO
    grad_clip: float = GRAD_CLIP
    num_workers: int = NUM_WORKERS
    num_labels: int = 2
    test_topic: str = 'gun control'
    seed: int = 0
    use_mtl: bool = False
    dip_max_examples: int | None = DIP_MAX_EXAMPLES
    dip_query_text_map: dict[str, str] = field(default_factory=lambda: QUERY_TEXT_MAP)

def run_tag(hp: HParams) -> str:
    mtl = 'mtl' if hp.use_mtl else 'single'
    return f"{mtl}_L{hp.num_labels}_{hp.test_topic.replace(' ', '_')}_seed{hp.seed}"

def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def _make_optim(model: nn.Module, hp: HParams, total_steps: int):
    no_decay = ('bias', 'LayerNorm.weight')
    groups = [
        {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
         'weight_decay': hp.weight_decay},
        {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
         'weight_decay': 0.0},
    ]
    optim = AdamW(groups, lr=hp.lr)
    sched = get_linear_schedule_with_warmup(optim, int(total_steps * hp.warmup_ratio), total_steps)
    return optim, sched

# --- Checkpointing ---------------------------------------------------------
# After every epoch we overwrite a single ckpt.pt with model+optim+sched+epoch
# +best-so-far. On restart, the run resumes at epoch+1 with the best weights
# already restored. The ckpt is ~1-1.5 GB (RoBERTa-base + AdamW state) and is
# deleted once the run finishes successfully and the final JSON is on Drive.

def _ckpt_path(hp: HParams) -> Path:
    return hp.output_dir / 'checkpoints' / run_tag(hp) / 'ckpt.pt'

def _save_ckpt(hp, model, optim, sched, epoch, best_val, best_state):
    path = _ckpt_path(hp)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.pt.tmp')
    torch.save({
        'model': model.state_dict(),
        'optim': optim.state_dict(),
        'sched': sched.state_dict(),
        'epoch': epoch,
        'best_val_loss': best_val,
        'best_state': best_state,
        'rng_torch': torch.get_rng_state(),
        'rng_cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        'rng_numpy': np.random.get_state(),
        'rng_python': random.getstate(),
    }, tmp)
    tmp.replace(path)  # atomic rename: a crash mid-save never corrupts ckpt

def _load_ckpt(hp, model, optim, sched):
    path = _ckpt_path(hp)
    if not path.exists():
        return 0, float('inf'), None
    data = torch.load(path, map_location=DEVICE)
    model.load_state_dict(data['model'])
    optim.load_state_dict(data['optim'])
    sched.load_state_dict(data['sched'])
    torch.set_rng_state(data['rng_torch'].cpu())
    if data.get('rng_cuda') is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([s.cpu() for s in data['rng_cuda']])
    np.random.set_state(data['rng_numpy'])
    random.setstate(data['rng_python'])
    start = data['epoch'] + 1
    print(f"  resuming {run_tag(hp)}: {start}/{hp.epochs} epochs done, "
          f"{hp.epochs - start} remaining (best val loss so far {data['best_val_loss']:.4f})")
    return start, data['best_val_loss'], data['best_state']

def _cleanup_ckpt(hp: HParams) -> None:
    path = _ckpt_path(hp)
    if path.exists():
        path.unlink()
    try:
        path.parent.rmdir()
    except OSError:
        pass

def _epoch_bar(hp: HParams, start_epoch: int):
    """tqdm over epochs, pre-filled with whatever was already completed."""
    return tqdm(
        total=hp.epochs,
        initial=start_epoch,
        desc=f'{run_tag(hp)} epochs',
        unit='ep',
        position=0,
        leave=True,
    )
# ---------------------------------------------------------------------------

@torch.no_grad()
def _evaluate(model, loader, mtl_task: str | None = None):
    model.eval()
    losses, preds, golds = [], [], []
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        if mtl_task is None:
            out = model(**batch); loss, logits = out.loss, out.logits
        else:
            out = model(**batch, task=mtl_task); loss, logits = out['loss'], out['logits']
        losses.append(loss.item() * batch['labels'].size(0))
        preds.append(logits.argmax(dim=-1).cpu().numpy())
        golds.append(batch['labels'].cpu().numpy())
    n = sum(len(g) for g in golds)
    return sum(losses) / max(n, 1), np.concatenate(preds), np.concatenate(golds)

def _train_single(hp: HParams) -> dict:
    splits = build_ukp_splits(hp.ukp_csv, hp.test_topic, hp.num_labels)
    model, tok = build_single_task_model(hp.model_name, hp.num_labels)
    model.to(DEVICE)
    coll = partial(collate, pad_token_id=tok.pad_token_id)
    train_loader = DataLoader(UKPDataset(splits['train'], tok, hp.max_length),
                              batch_size=hp.batch_size, shuffle=True,
                              collate_fn=coll, num_workers=hp.num_workers)
    val_loader = DataLoader(UKPDataset(splits['val'], tok, hp.max_length),
                            batch_size=hp.eval_batch_size, collate_fn=coll)
    test_loader = DataLoader(UKPDataset(splits['test'], tok, hp.max_length),
                             batch_size=hp.eval_batch_size, collate_fn=coll)
    optim, sched = _make_optim(model, hp, len(train_loader) * hp.epochs)
    start_epoch, best_val, best_state = _load_ckpt(hp, model, optim, sched)
    ebar = _epoch_bar(hp, start_epoch)
    for epoch in range(start_epoch, hp.epochs):
        model.train()
        pbar = tqdm(train_loader, desc=f'ep {epoch+1}/{hp.epochs}',
                    position=1, leave=False)
        for batch in pbar:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(**batch); out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), hp.grad_clip)
            optim.step(); sched.step(); optim.zero_grad()
            pbar.set_postfix(loss=float(out.loss.item()))
        val_loss, _, _ = _evaluate(model, val_loader)
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        _save_ckpt(hp, model, optim, sched, epoch, best_val, best_state)
        ebar.set_postfix(val_loss=f'{val_loss:.4f}', best=f'{best_val:.4f}')
        ebar.update(1)
    ebar.close()
    if best_state is not None:
        model.load_state_dict(best_state)
    _, preds, golds = _evaluate(model, test_loader)
    return compute_metrics(golds, preds, hp.num_labels) | {'best_val_loss': best_val}

def _train_mtl(hp: HParams) -> dict:
    splits = build_ukp_splits(hp.ukp_csv, hp.test_topic, hp.num_labels)
    model, tok = build_mtl_model(hp.model_name, hp.num_labels)
    model.to(DEVICE)
    coll = partial(collate, pad_token_id=tok.pad_token_id)
    train_loader = DataLoader(UKPDataset(splits['train'], tok, hp.max_length),
                              batch_size=hp.batch_size, shuffle=True,
                              collate_fn=coll, num_workers=hp.num_workers)
    val_loader = DataLoader(UKPDataset(splits['val'], tok, hp.max_length),
                            batch_size=hp.eval_batch_size, collate_fn=coll)
    test_loader = DataLoader(UKPDataset(splits['test'], tok, hp.max_length),
                             batch_size=hp.eval_batch_size, collate_fn=coll)
    dip_examples = load_dip2016(hp.dip_dir,
                                query_text_map=hp.dip_query_text_map or None,
                                max_examples=hp.dip_max_examples)
    aux_loader = DataLoader(DIPDataset(dip_examples, tok, hp.max_length),
                            batch_size=hp.batch_size, shuffle=True,
                            collate_fn=coll, num_workers=hp.num_workers)
    optim, sched = _make_optim(model, hp, (len(train_loader) + len(aux_loader)) * hp.epochs)
    start_epoch, best_val, best_state = _load_ckpt(hp, model, optim, sched)
    ebar = _epoch_bar(hp, start_epoch)
    def _run_epoch(loader, task, tag):
        model.train()
        pbar = tqdm(loader, desc=tag, position=1, leave=False)
        for batch in pbar:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(**batch, task=task); out['loss'].backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), hp.grad_clip)
            optim.step(); sched.step(); optim.zero_grad()
            pbar.set_postfix(loss=float(out['loss'].item()))
    for epoch in range(start_epoch, hp.epochs):
        _run_epoch(aux_loader,   'aux',  f'ep {epoch+1}/{hp.epochs} [aux]')
        _run_epoch(train_loader, 'main', f'ep {epoch+1}/{hp.epochs} [main]')
        val_loss, _, _ = _evaluate(model, val_loader, mtl_task='main')
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        _save_ckpt(hp, model, optim, sched, epoch, best_val, best_state)
        ebar.set_postfix(val_loss=f'{val_loss:.4f}', best=f'{best_val:.4f}')
        ebar.update(1)
    ebar.close()
    if best_state is not None:
        model.load_state_dict(best_state)
    _, preds, golds = _evaluate(model, test_loader, mtl_task='main')
    return compute_metrics(golds, preds, hp.num_labels) | {'best_val_loss': best_val}

def run_one(hp: HParams) -> dict:
    # Only seed when starting fresh; a resumed run restores RNG from the ckpt.
    if not _ckpt_path(hp).exists():
        set_seed(hp.seed)
    metrics = _train_mtl(hp) if hp.use_mtl else _train_single(hp)
    metrics.update({
        'test_topic': hp.test_topic,
        'seed': hp.seed,
        'num_labels': hp.num_labels,
        'use_mtl': hp.use_mtl,
        'model_name': hp.model_name,
    })
    _cleanup_ckpt(hp)  # final JSON is the source of truth; drop the ~1GB ckpt
    return metrics

## L. Sanity check

In [ ]:
df = load_ukp_csv(UKP_CSV)
print('UKP rows:', len(df))
print('UKP topics:', sorted(df['topic_norm'].unique()))
print(df.groupby(['topic_norm', 'set', 'annotation']).size().unstack(fill_value=0).head(20))

dip_sample = load_dip2016(DIP_DIR, query_text_map=QUERY_TEXT_MAP or None, limit_files=3)
print(f'\nDIP (first 3 files): {len(dip_sample)} sentences')
for ex in dip_sample[:3]:
    print(f'  query={ex.query!r}  label={ex.label}  sent={ex.sentence[:80]!r}')

## M. Smoke test  (1 topic, 1 seed, 2 epochs, single-task)

~3–5 min on a T4 GPU. Confirms the pipeline runs end-to-end before launching the long grid.

In [ ]:
hp_smoke = HParams(epochs=2, test_topic='gun control', num_labels=2, seed=0, use_mtl=False)
smoke_res = run_one(hp_smoke)
print(smoke_res)

## N. Full grid  (resumable at run-level **and** epoch-level)

Iterates `{single, MTL} × {2-label, 3-label} × all topics × all seeds`.

- A finished run is saved to `OUTPUT_DIR/<tag>.json` and skipped on rerun.
- An in-flight run writes `OUTPUT_DIR/checkpoints/<tag>/ckpt.pt` after each epoch. If Colab dies mid-epoch, just rerun this cell — the run picks up at the next epoch with optimizer / scheduler / RNG restored.

⚠️ The full grid (10 seeds × 8 topics × 2 setups × 2 modes = 320 runs) takes days on a single T4. Trim `SEEDS`, `LABEL_SETUPS`, `MTL_MODES`, or `TEST_TOPICS` in **Section C** before launching.

In [ ]:
topics_to_run = TEST_TOPICS or TOPICS

# Build the full list of (use_mtl, num_labels, topic, seed) combinations.
plan = [
    (use_mtl, num_labels, topic, seed)
    for use_mtl in MTL_MODES
    for num_labels in LABEL_SETUPS
    for topic in topics_to_run
    for seed in SEEDS
]
def _tag(use_mtl, num_labels, topic, seed):
    return f"{'mtl' if use_mtl else 'single'}_L{num_labels}_{topic.replace(' ', '_')}_seed{seed}"

done_at_start = sum(
    1 for combo in plan
    if (OUTPUT_DIR / f'{_tag(*combo)}.json').exists()
)
print(f'Grid: {len(plan)} runs total, {done_at_start} already done on Drive, '
      f'{len(plan) - done_at_start} remaining.')

# Manual tqdm so cached runs (counted in `initial`) don't double-advance the bar.
grid_bar = tqdm(total=len(plan), initial=done_at_start,
                desc='grid', unit='run', position=2, leave=True)

results: dict[tuple, list[dict]] = {}
for combo in plan:
    use_mtl, num_labels, topic, seed = combo
    tag = _tag(*combo)
    out_path = OUTPUT_DIR / f'{tag}.json'
    grid_bar.set_postfix_str(tag)
    if out_path.exists():
        res = json.loads(out_path.read_text())
        # already counted in done_at_start; do not advance
    else:
        hp = HParams(test_topic=topic, num_labels=num_labels,
                     seed=seed, use_mtl=use_mtl)
        res = run_one(hp)
        out_path.write_text(json.dumps(res, indent=2))
        grid_bar.update(1)
    results.setdefault((use_mtl, num_labels, topic), []).append(res)
grid_bar.close()

# Aggregate per (mode, labels) and overall.
summary: dict = {}
seen_modes = list({(m, n) for m, n, _, _ in plan})
for use_mtl, num_labels in seen_modes:
    key = f"{'mtl' if use_mtl else 'single'}_{num_labels}label"
    per_topic = {t: results[(use_mtl, num_labels, t)] for t in topics_to_run}
    summary[key] = {t: aggregate_runs(r) for t, r in per_topic.items()}
    all_runs = [r for rs in per_topic.values() for r in rs]
    summary[key]['__overall__'] = aggregate_runs(all_runs)

(OUTPUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2))
print('\nSaved summary to', OUTPUT_DIR / 'summary.json')

## O. Inspect summary

In [ ]:
summary = json.loads((OUTPUT_DIR / 'summary.json').read_text())
for setup, by_topic in summary.items():
    print(f'\n=== {setup} ===')
    overall = by_topic['__overall__']
    for k, (m, s) in overall.items():
        print(f'  {k:<14s} {m:.4f} ± {s:.4f}')